In [ ]:
import papermill as pm
import numpy as np
from jaxcmr.helpers import find_project_root, load_data
from IPython.display import Image, display
import os

# Render Model Fitting: Strength Models

Orchestrate subject-level fitting notebooks for the no-context strength baselines.

In [ ]:
from lpp_ecmr.fitting_config import BASE_PARAMS
from lpp_ecmr.strength_registry import STRENGTH_MODELS

base_params = BASE_PARAMS
prepare_only = True

In [ ]:
varied_parameters = [m for m in STRENGTH_MODELS if m.get("enabled")]
print(f"{len(varied_parameters)} models enabled")

In [ ]:
project_root = find_project_root()
workspace = os.path.dirname(project_root)
fitting_template = os.path.join(workspace, "jaxcmr", "templates", "fitting.ipynb")
simulation_template = os.path.join(workspace, "jaxcmr", "templates", "simulation.ipynb")
rendered_dir = os.path.join(project_root, "analyses", "rendered")
os.makedirs(rendered_dir, exist_ok=True)

data = load_data(os.path.join(project_root, base_params["data_path"]))
n_subjects = np.unique(data["subject"].flatten()).shape[0]
print(f"{n_subjects} subjects")

for partial_params in varied_parameters:

    params = base_params.copy() | {
        k: v for k, v in partial_params.items() if k != "enabled"
    }

    data_tag = params["data_tag"]
    model_name = params["model_name"]
    base_run_tag = params["base_run_tag"]
    best_of = params["best_of"]
    fit_tag = f"{base_run_tag}_best_of_{best_of}"
    max_subjects = params["max_subjects"]
    if max_subjects:
        fit_tag += f"_nsubs_{max_subjects}"
    fit_stem = f"{data_tag}_{model_name}_{fit_tag}"

    # Per-subject fitting notebooks
    for subj_idx in range(n_subjects):
        template_params = params | {
            "trial_query": params["trial_query"],
            "target_directory": "",
            "figure_dir": "figures/fitting",
            "figure_str": f"{fit_stem}_sub{subj_idx}",
            "display_iterations": False,
            "subject_indices": [subj_idx],
            "redo_sims": False,
            "single_analysis_configs": [],
        }

        output_path = os.path.join(
            rendered_dir,
            f"fitting_{fit_stem}_sub{subj_idx}.ipynb",
        )

        pm.execute_notebook(
            fitting_template,
            output_path,
            parameters=template_params,
            progress_bar=True,
            prepare_only=prepare_only,
        )

    # Simulation notebook (runs after merge)
    simulation_params = {
        "data_path": params["data_path"],
        "fit_path": f"fits/{fit_stem}.json",
        "target_directory": "",
        "experiment_count": params["experiment_count"],
        "max_subjects": 0,
        "make_factory_path": params["make_factory_path"],
        "component_paths": params.get("component_paths", base_params["component_paths"]),
        "sim_alg_path": params.get("sim_alg_path", base_params["sim_alg_path"]),
        "pooled": False,
        "filter_repeated_recalls": params.get("filter_repeated_recalls", True),
        "redo_sims": False,
        "redo_figures": False,
        "trial_query": params["trial_query"],
        "seed": 0,
        "comparison_analysis_configs": params["comparison_analysis_configs"],
        "single_analysis_configs": params["single_analysis_configs"],
        "figure_dir": "figures/fitting",
        "figure_str": fit_stem,
    }

    sim_output = os.path.join(rendered_dir, f"simulation_{fit_stem}.ipynb")
    pm.execute_notebook(
        simulation_template,
        sim_output,
        parameters=simulation_params,
        progress_bar=True,
        prepare_only=prepare_only,
    )

    print(f"{model_name}: {n_subjects} fitting + 1 simulation notebook")